# Multivariate Linear Regression — Q-CHAT-10 Toddler Autism Screening

**Mission:** predict the Q-CHAT-10 screening score from the full behavioral checklist
(A1-A10) plus demographics, to help underserved families and clinics get quick severity
estimates without waiting for a specialist.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib

sns.set_style('whitegrid')

## 1. Load data

In [2]:
df = pd.read_csv('Toddler Autism dataset July 2018.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1054 entries, 0 to 1053
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Case_No                 1054 non-null   int64
 1   A1                      1054 non-null   int64
 2   A2                      1054 non-null   int64
 3   A3                      1054 non-null   int64
 4   A4                      1054 non-null   int64
 5   A5                      1054 non-null   int64
 6   A6                      1054 non-null   int64
 7   A7                      1054 non-null   int64
 8   A8                      1054 non-null   int64
 9   A9                      1054 non-null   int64
 10  A10                     1054 non-null   int64
 11  Age_Mons                1054 non-null   int64
 12  Qchat-10-Score          1054 non-null   int64
 13  Sex                     1054 non-null   str  
 14  Ethnicity               1054 non-null   str  
 15  Jaundice                1054 non

In [3]:
df.isnull().sum()

Case_No                   0
A1                        0
A2                        0
A3                        0
A4                        0
A5                        0
A6                        0
A7                        0
A8                        0
A9                        0
A10                       0
Age_Mons                  0
Qchat-10-Score            0
Sex                       0
Ethnicity                 0
Jaundice                  0
Family_mem_with_ASD       0
Who completed the test    0
Class/ASD Traits          0
dtype: int64

## 2. Drop leakage / non-predictive columns

Drops only `Case_No` (an identifier, not a predictor) and `Class/ASD Traits ` (a direct
threshold of Qchat-10-Score, so it's leakage). Everything else — including `A1`-`A10` and
`Who completed the test` — stays, since the mission calls for using the full behavioral
checklist.

In [4]:
df = df.drop(columns=['Case_No', 'Class/ASD Traits '])
df.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age_Mons,Qchat-10-Score,Sex,Ethnicity,Jaundice,Family_mem_with_ASD,Who completed the test
0,0,0,0,0,0,0,1,1,0,1,28,3,f,middle eastern,yes,no,family member
1,1,1,0,0,0,1,1,0,0,0,36,4,m,White European,yes,no,family member
2,1,0,0,0,0,0,1,1,0,1,36,4,m,middle eastern,yes,no,family member
3,1,1,1,1,1,1,1,1,1,1,24,10,m,Hispanic,no,no,family member
4,1,1,0,1,1,1,1,1,1,1,20,9,f,White European,no,yes,family member
